# Machine learning part of my MSc thesis

This notebook has the machine learning part of my dissertation *"Machine Learning versus Structural Gravity: Predicting and Explaining EU Import Flows of Textile Trimmings, with an Application to Brazilian Suppliers (2015-2025)"* (Gisma University of Applied Sciences, 2026).

The question here (RQ1 of the thesis): can ML models predict EU imports of textile trimmings at product level better than a structural gravity model? I keep the notebook short on purpose. The full pipeline with the data downloads, the PPML gravity estimation, significance tests and SHAP is in my repo: https://github.com/THS-99/trimmings-gravity-vs-ml

One honest note: this is a simplified version of scripts 07-10 from the repo, so the numbers here can differ a little from the exact ones in the thesis tables. The whole notebook takes around 10 minutes to run on a normal Colab CPU runtime.

In [ ]:
# everything used here is preinstalled on Colab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42

## 1. Load the data

The panel I built has one row per exporter x EU destination x HS6 product x year, 2015 to 2025, with zeros kept as real zeros (no trade that year). It sits compressed in my repo, so it loads with one line.

In [ ]:
url = "https://raw.githubusercontent.com/THS-99/trimmings-gravity-vs-ml/main/data/processed/panel_trimmings_2015_2025.csv.gz"
df = pd.read_csv(url, dtype={"hs6": str, "heading": str})
print(df.shape)
df.head()

In [ ]:
# quick sanity checks
print("years:", df.year.min(), "to", df.year.max())
print("exporters:", df.exporter.nunique(), "| destinations:", df.destination.nunique(), "| products:", df.hs6.nunique())
print("share of zero flows:", round((df.value_eur == 0).mean(), 3))

## 2. Features

Trade flows are very persistent: a flow that was big last year is probably big this year too. So besides the gravity variables (GDP, population, distance, common language and so on) I give the models the recent history of each flow: the last two values, a 3-year rolling mean (all in logs) and whether the flow was zero last year.

The target is log1p of the import value, because the values are extremely skewed and there are lots of zeros. The exporter, destination and product identities go in as one-hot dummies.

In [ ]:
df = df.sort_values(["exporter", "destination", "hs6", "year"]).reset_index(drop=True)
df["log_value"] = np.log1p(df["value_eur"])

grp = df.groupby(["exporter", "destination", "hs6"])["log_value"]
df["lag1"] = grp.shift(1)
df["lag2"] = grp.shift(2)
df["roll3"] = grp.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
df["was_zero_last_year"] = (df["lag1"] == 0).astype(int)

# the first years of each flow have no history, I just fill with 0 (log1p(0) is also 0)
hist_cols = ["lag1", "lag2", "roll3", "was_zero_last_year"]
df[hist_cols] = df[hist_cols].fillna(0)

In [ ]:
# log the skewed gravity variables, keep the dummies as they are
for c in ["dist", "gdp_exporter", "pop_exporter", "gdp_destination", "pop_destination"]:
    df["log_" + c] = np.log(df[c])

gravity_cols = ["log_dist", "contig", "comlang_off", "comcol", "rta",
                "log_gdp_exporter", "log_pop_exporter", "log_gdp_destination", "log_pop_destination"]

X_full = pd.get_dummies(df[gravity_cols + hist_cols + ["exporter", "destination", "hs6"]],
                        columns=["exporter", "destination", "hs6"])
y = df["log_value"]
years = df["year"]
print(X_full.shape)

## 3. How I evaluate

The models never see the test year in training. I use three splits ("rolling origin"): train up to 2022 and test on 2023, train up to 2023 and test on 2024, train up to 2024 and test on 2025.

Predictions are made in logs but turned back into euros with expm1, and the errors (RMSE and MAE) are measured in euros, because that is the unit that actually matters.

In [ ]:
def rmse_mae_in_euros(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)  # no negative trade
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mae

## 4. Models

Four things get compared:

1. a naive baseline that just repeats last year's value (surprisingly hard to beat in trade data),
2. a random forest,
3. LightGBM,
4. the same random forest but *without* the history features, to see how much of the performance actually comes from them.

The hyperparameters are the winners of a small grid search I ran separately (script 09 and `results/ml_tuning_report.json` in the repo), I just reuse them here.

In [ ]:
test_years = [2023, 2024, 2025]
results = []

X_nohist = X_full.drop(columns=hist_cols)

for test_year in test_years:
    train = years < test_year
    test = years == test_year

    # 1. naive baseline: last year's value
    rmse, mae = rmse_mae_in_euros(y[test], df.loc[test, "lag1"])
    results.append(["naive (last year)", test_year, rmse, mae])

    # 2. random forest
    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                               max_samples=0.7, random_state=SEED, n_jobs=-1)
    rf.fit(X_full[train], y[train])
    rmse, mae = rmse_mae_in_euros(y[test], rf.predict(X_full[test]))
    results.append(["random forest", test_year, rmse, mae])

    # 3. LightGBM
    lgbm = LGBMRegressor(n_estimators=1500, learning_rate=0.05, num_leaves=63,
                         random_state=SEED, n_jobs=-1, verbose=-1)
    lgbm.fit(X_full[train], y[train])
    rmse, mae = rmse_mae_in_euros(y[test], lgbm.predict(X_full[test]))
    results.append(["lightgbm", test_year, rmse, mae])

    # 4. random forest without the history features
    rf_nohist = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                      max_samples=0.7, random_state=SEED, n_jobs=-1)
    rf_nohist.fit(X_nohist[train], y[train])
    rmse, mae = rmse_mae_in_euros(y[test], rf_nohist.predict(X_nohist[test]))
    results.append(["random forest, no history", test_year, rmse, mae])

    print("done with", test_year)

In [ ]:
res = pd.DataFrame(results, columns=["model", "test_year", "rmse_eur", "mae_eur"])
res.pivot(index="model", columns="test_year", values="rmse_eur").round(0)

In [ ]:
# average over the three test years
res.groupby("model")[["rmse_eur", "mae_eur"]].mean().round(0).sort_values("rmse_eur")

For reference: the PPML gravity model of the thesis (script 08 in the repo) gets an average out-of-sample RMSE of about EUR 335,000 on these same three test years. So the random forest with the history features roughly halves the gravity error, but the same forest *without* them is no better than gravity.

Two other things I find worth noticing in the table. The naive baseline is already very strong, almost as good as the forest, which says a lot about how persistent these flows are. And 2024 is the hardest year for everyone, there were some large swings in that year that no model saw coming.

## 5. Which features matter

A quick look at what the random forest actually uses (the last one trained, so up to 2024). Not a deep analysis, the thesis uses permutation importance and SHAP for that, but the picture is already clear.

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X_full.columns).sort_values().tail(15)
imp.plot.barh(figsize=(7, 5))
plt.title("Random forest feature importance (top 15)")
plt.tight_layout()
plt.show()

## 6. What I take from this

The importance plot is dominated by the history features (the 3-year rolling mean above everything else), with GDP and the identity dummies far behind. Same message as the ablation: the machine learning models win because trade flows repeat themselves, not because they use the gravity variables in a smarter way. That is the answer this notebook gives to RQ1, and it matches the full pipeline in the repo, where the difference is also tested formally (paired bootstrap, p < 0.001).

What is *not* in this notebook: the PPML gravity estimation itself, the significance tests, permutation importance and SHAP (RQ2), and the Brazil under-trading analysis (RQ3). All of that is in the repo, and the discussion is in chapters 4 and 5 of the thesis.

https://github.com/THS-99/trimmings-gravity-vs-ml